# Lab 2 — SVD, PCA, and the Geometry of Neural Representations

**Linear Algebra · Upskilled**

SVD is the factorisation that sits beneath PCA, low-rank approximation, pseudo-inverses, and LoRA. In this lab you will:

1. Compute the **full SVD** of a GloVe slice and read its geometric meaning from the singular values.
2. Build **low-rank approximations** at ranks 1, 5, 10, 25, 50 and measure reconstruction error.
3. Implement **PCA from scratch** via SVD, compare to `sklearn`, and visualise principal components.
4. Compute **effective rank** and understand why it matters for neural network weight matrices.
5. Implement SVD in both **PyTorch** (`torch.linalg.svd`) and **TensorFlow** (`tf.linalg.svd`) and compare.
6. Explore **implicit low-rank regularisation**: why gradient descent on overparameterised models biases toward low-rank solutions.

---
> **Colab tip**: Runtime → Run all  (Ctrl+F9). GPU is optional but useful for the full-matrix SVD.

## 0 · Setup

In [ ]:
import subprocess, sys
def pip(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

pip('torch'); pip('tensorflow'); pip('numpy'); pip('matplotlib'); pip('scikit-learn'); pip('requests')

import numpy as np
import torch
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA as skPCA
import requests, zipfile, io, os

print('torch :', torch.__version__)
print('tf    :', tf.__version__)

In [ ]:
# ── Load GloVe 50d ───────────────────────────────────────────────────────────
GLOVE_PATH = 'glove.6B.50d.txt'

if not os.path.exists(GLOVE_PATH):
    print('Downloading GloVe 6B 50d …')
    r = requests.get('https://nlp.stanford.edu/data/glove.6B.zip', stream=True)
    zipfile.ZipFile(io.BytesIO(r.content)).extract('glove.6B.50d.txt', '.')
    print('Done.')

words, vecs = [], []
with open(GLOVE_PATH, encoding='utf-8') as f:
    for line in f:
        parts = line.rstrip().split(' ')
        words.append(parts[0])
        vecs.append(list(map(float, parts[1:])))

word2idx = {w: i for i, w in enumerate(words)}
E = np.array(vecs, dtype=np.float32)   # (400000, 50)
print(f'E shape: {E.shape}')

---
## 1 · The Full SVD

Every matrix $A \in \mathbb{R}^{m \times n}$ (with $m \geq n$) factors as:
$$A = U \Sigma V^T$$
where $U \in \mathbb{R}^{m \times m}$ and $V \in \mathbb{R}^{n \times n}$ are orthogonal, and $\Sigma$ is diagonal with non-negative entries $\sigma_1 \geq \sigma_2 \geq \cdots \geq \sigma_n \geq 0$.

We work on a 5 000 × 50 slice to keep runtimes short.

In [ ]:
# ── 1a. Slice: top 5000 words ────────────────────────────────────────────────
A = E[:5000].copy()   # (5000, 50)  — mean-centre first for PCA compatibility
A_mean = A.mean(axis=0, keepdims=True)
A_c = A - A_mean      # mean-centred

# Full SVD
U, S, Vh = np.linalg.svd(A_c, full_matrices=False)  # economy SVD
# U: (5000, 50)  S: (50,)  Vh: (50, 50)

print(f'U  shape: {U.shape}')
print(f'S  shape: {S.shape}')
print(f'Vh shape: {Vh.shape}')
print(f'Top-10 singular values: {np.round(S[:10], 2)}')

# Verify reconstruction: A_c ≈ U @ diag(S) @ Vh
A_recon = U * S @ Vh   # broadcasting: (U * S) multiplies each column of U by s_i
print(f'Max reconstruction error: {np.abs(A_c - A_recon).max():.2e}')

In [ ]:
# ── 1b. Singular value spectrum plot ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(range(1, len(S)+1), S, color='#2a5757', width=0.8)
axes[0].set_xlabel('singular value index')
axes[0].set_ylabel('sigma_i')
axes[0].set_title('Singular value spectrum')

# Cumulative explained variance
var_explained = (S**2) / (S**2).sum()
cumvar = np.cumsum(var_explained)
axes[1].plot(range(1, len(S)+1), cumvar * 100, color='#2a5757', lw=2)
axes[1].axhline(90, ls='--', color='#63aac9', label='90%')
axes[1].axhline(95, ls=':', color='#63aac9', label='95%')
axes[1].set_xlabel('number of components')
axes[1].set_ylabel('cumulative variance explained (%)')
axes[1].set_title('Cumulative variance')
axes[1].legend()

plt.tight_layout()
plt.show()

k90 = int(np.searchsorted(cumvar, 0.90)) + 1
k95 = int(np.searchsorted(cumvar, 0.95)) + 1
print(f'Components to explain 90% variance: {k90}')
print(f'Components to explain 95% variance: {k95}')

**Reflection 1a**: The singular value spectrum decays rapidly. How does this fast decay relate to the success of dimensionality reduction in NLP? If the spectrum were flat (all singular values equal), what would that imply about the structure of the data?

---
## 2 · Low-Rank Approximation

The **Eckart-Young theorem** states that the best rank-$k$ approximation to $A$ in both Frobenius and spectral norm is:
$$A_k = \sum_{i=1}^{k} \sigma_i \vec{u}_i \vec{v}_i^T = U_k \Sigma_k V_k^T$$

This is exactly what LoRA does to neural network weight matrices.

In [ ]:
# ── 2a. Low-rank approximation at multiple ranks ──────────────────────────────
ranks = [1, 5, 10, 25, 50]
frob_total = np.linalg.norm(A_c, 'fro')

print(f'{"rank k":10s}  {"Frob error":15s}  {"rel error":12s}  {"% variance captured"}')
print('-' * 62)
for k in ranks:
    A_k = U[:, :k] * S[:k] @ Vh[:k, :]            # rank-k reconstruction
    err = np.linalg.norm(A_c - A_k, 'fro')
    # Parseval: sum of squared dropped singular values = squared Frobenius error
    var_cap = (S[:k]**2).sum() / (S**2).sum() * 100
    print(f'{k:<10d}  {err:<15.4f}  {err/frob_total:<12.4f}  {var_cap:.2f}%')

In [ ]:
# ── 2b. Nearest-word quality at different ranks ───────────────────────────────
def nearest_in(matrix, query_word, top_k=5):
    """Find nearest words in a given embedding matrix."""
    idx = word2idx[query_word]
    q = matrix[idx]
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    unit  = matrix / np.clip(norms, 1e-9, None)
    scores = unit @ (q / np.linalg.norm(q))
    order  = np.argsort(-scores)
    results = []
    for i in order:
        if words[i] != query_word:
            results.append((words[i], float(scores[i])))
        if len(results) == top_k:
            break
    return results

# Restrict to the 5000-word slice for comparison
A_orig_slice = A   # original (not centred)
probe = 'science'

for k in [1, 5, 10, 50]:
    A_k = U[:, :k] * S[:k] @ Vh[:k, :] + A_mean  # add mean back
    neighbours = nearest_in(A_k, probe, top_k=4)
    print(f'k={k:2d}: {[w for w,_ in neighbours]}')

**Reflection 2a**: At rank 1 the neighbourhood collapses to a single dominant direction. What does this direction likely represent? At what rank do the neighbours start to look semantically reasonable?

---
## 3 · PCA from Scratch

PCA = SVD of the mean-centred data matrix. The principal components are the right singular vectors $V^T$; the principal-component scores (coordinates in PC space) are $U \Sigma$.

In [ ]:
# ── 3a. PCA via SVD ───────────────────────────────────────────────────────────
# Already have U, S, Vh from the full SVD of A_c

k_pca = 2
# PC scores: projection of each word onto first k principal axes
scores = U[:, :k_pca] * S[:k_pca]          # (5000, 2)  [= A_c @ Vh[:k].T]

# Verify: equivalent to A_c @ Vh[:k_pca].T
scores_alt = A_c @ Vh[:k_pca].T
print(f'Max diff between methods: {np.abs(scores - scores_alt).max():.2e}')

# ── 3b. Compare to sklearn ────────────────────────────────────────────────────
pca_sk = skPCA(n_components=k_pca)
scores_sk = pca_sk.fit_transform(A_c)

# sklearn may flip sign conventions — compare absolute correlation
corr0 = np.corrcoef(scores[:, 0], scores_sk[:, 0])[0, 1]
corr1 = np.corrcoef(scores[:, 1], scores_sk[:, 1])[0, 1]
print(f'PC1 correlation (ours vs sklearn): {abs(corr0):.6f}')
print(f'PC2 correlation (ours vs sklearn): {abs(corr1):.6f}')

In [ ]:
# ── 3c. Visualise select word clusters in PC space ───────────────────────────
groups = {
    'royalty':   ['king', 'queen', 'prince', 'princess', 'throne'],
    'geography': ['france', 'germany', 'spain', 'italy', 'japan'],
    'science':   ['physics', 'chemistry', 'biology', 'math', 'science'],
    'food':      ['bread', 'cheese', 'wine', 'meat', 'fruit'],
}
colours = ['#2a5757', '#63aac9', '#e07b39', '#8b5e3c']

fig, ax = plt.subplots(figsize=(9, 7))
for (group, wlist), col in zip(groups.items(), colours):
    for w in wlist:
        if w in word2idx and word2idx[w] < 5000:
            i = word2idx[w]
            ax.scatter(scores[i, 0], scores[i, 1], color=col, s=40)
            ax.annotate(w, (scores[i, 0], scores[i, 1]), fontsize=8)
    ax.scatter([], [], color=col, label=group, s=60)

ax.set_xlabel(f'PC1 ({pca_sk.explained_variance_ratio_[0]*100:.1f}% var)')
ax.set_ylabel(f'PC2 ({pca_sk.explained_variance_ratio_[1]*100:.1f}% var)')
ax.set_title('GloVe top-5000 words in principal-component space')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

**Reflection 3a**: Do the word groups cluster in PC space? What do PC1 and PC2 seem to encode? PCA is unsupervised — it maximises variance, not semantic coherence. Why do semantic groups appear?

---
## 4 · Effective Rank

The **effective rank** of a matrix is a soft measure of how many singular values carry significant information. The most common definition uses the normalised singular-value entropy:
$$\text{erank}(A) = \exp\!\left(-\sum_{i} p_i \ln p_i\right), \quad p_i = \frac{\sigma_i}{\sum_j \sigma_j}$$

This equals the number of non-zero singular values when the spectrum is flat; it is 1 when only one singular value is non-zero.

In [ ]:
# ── 4a. Effective rank of GloVe slice ─────────────────────────────────────────
def effective_rank(singular_values):
    s = singular_values[singular_values > 1e-9]
    p = s / s.sum()
    entropy = -np.sum(p * np.log(p))
    return float(np.exp(entropy))

er = effective_rank(S)
print(f'Effective rank of 5000x50 GloVe slice: {er:.2f}  (max possible = {len(S)})')

# Compare: random Gaussian matrix of same shape
rng = np.random.default_rng(0)
R = rng.standard_normal(A_c.shape).astype(np.float32)
_, S_rand, _ = np.linalg.svd(R, full_matrices=False)
er_rand = effective_rank(S_rand)
print(f'Effective rank of random matrix:        {er_rand:.2f}')

In [ ]:
# ── 4b. Effective rank of neural network weight matrices ──────────────────────
# Simulate weight matrices at different stages of training:
# early (random init), mid (partially trained), late (trained, more structured)

sizes = (512, 256)
rng = np.random.default_rng(42)

# Random init
W_init = rng.standard_normal(sizes).astype(np.float32) * 0.02

# Low-rank signal + noise (simulates learned weight matrix)
k_signal = 16
U_s, _, Vh_s = np.linalg.svd(W_init, full_matrices=False)
W_trained  = U_s[:, :k_signal] @ (np.diag(np.linspace(5, 0.5, k_signal))) @ Vh_s[:k_signal]
W_trained += rng.standard_normal(sizes).astype(np.float32) * 0.005

for label, W in [('random init', W_init), ('trained weight', W_trained)]:
    _, Sw, _ = np.linalg.svd(W, full_matrices=False)
    print(f'{label:20s}  erank = {effective_rank(Sw):.2f}  |  top-5 svs: {np.round(Sw[:5], 3)}')

**Reflection 4a**: Trained weight matrices have lower effective rank than random initialisations. Why? How does this motivate LoRA (Low-Rank Adaptation), which only trains two small factor matrices $A \in \mathbb{R}^{d \times r}$ and $B \in \mathbb{R}^{r \times k}$ with $r \ll \min(d,k)$?

---
## 5 · SVD in PyTorch and TensorFlow

Both frameworks expose SVD natively; the API conventions differ slightly.

In [ ]:
# ── 5a. PyTorch SVD ───────────────────────────────────────────────────────────
A_t = torch.from_numpy(A_c)

# torch.linalg.svd returns (U, S, Vh)  — Vh is already V^T
U_t, S_t, Vh_t = torch.linalg.svd(A_t, full_matrices=False)
print('PyTorch:')
print(f'  U  : {tuple(U_t.shape)}')
print(f'  S  : {tuple(S_t.shape)}')
print(f'  Vh : {tuple(Vh_t.shape)}')
print(f'  Top-5 singular values: {S_t[:5].numpy().round(2)}')

# Reconstruct and verify
A_recon_t = U_t * S_t @ Vh_t
print(f'  Recon error: {(A_t - A_recon_t).abs().max().item():.2e}')

In [ ]:
# ── 5b. TensorFlow SVD ───────────────────────────────────────────────────────
A_tf = tf.constant(A_c)

# tf.linalg.svd returns (S, U, V)  — note: V, not Vh!
S_tf, U_tf, V_tf = tf.linalg.svd(A_tf, full_matrices=False)
print('TensorFlow:')
print(f'  S  : {tuple(S_tf.shape)}')
print(f'  U  : {tuple(U_tf.shape)}')
print(f'  V  : {tuple(V_tf.shape)}  (NOTE: this is V, not Vh)')
print(f'  Top-5 singular values: {S_tf[:5].numpy().round(2)}')

# Reconstruct: A ≈ U @ diag(S) @ V^T   (need to transpose V)
A_recon_tf = U_tf * S_tf @ tf.transpose(V_tf)
print(f'  Recon error: {float(tf.reduce_max(tf.abs(A_tf - A_recon_tf)).numpy()):.2e}')

In [ ]:
# ── 5c. Cross-framework consistency ──────────────────────────────────────────
# Singular values should match across NumPy, PyTorch, TF (up to float32 precision)
diff_pt = np.abs(S - S_t.numpy()).max()
diff_tf = np.abs(S - S_tf.numpy()).max()
print(f'Max |S_numpy - S_torch|: {diff_pt:.2e}')
print(f'Max |S_numpy - S_tf|   : {diff_tf:.2e}')

**Key API difference**: `torch.linalg.svd` returns `(U, S, Vh)` with $V^H$ (conjugate transpose). `tf.linalg.svd` returns `(S, U, V)` with $V$ (not transposed). Always check the docs when switching frameworks.

---
## 6 · Implicit Low-Rank Regularisation

When gradient descent trains an overparameterised matrix factorisation $W = AB$ (with $A \in \mathbb{R}^{m \times r}$, $B \in \mathbb{R}^{r \times n}$), it implicitly biases toward **low nuclear norm** (sum of singular values) — even without explicit regularisation. This was formalised by Gunasekar et al. (2017) and explains why LoRA works: the parameterisation itself acts as regularisation.

In [ ]:
# ── 6a. Matrix completion via gradient descent ────────────────────────────────
# Task: recover a low-rank matrix from a subset of its entries
# The solution found by gradient descent tends to have low nuclear norm

torch.manual_seed(0)
m, n, r_true = 50, 50, 3

# Ground truth: rank-3 matrix
U_gt = torch.randn(m, r_true) / r_true**0.5
V_gt = torch.randn(n, r_true) / r_true**0.5
W_gt = U_gt @ V_gt.T         # (50, 50), rank 3

# Observe 40% of entries
mask = torch.rand(m, n) < 0.4
print(f'Fraction observed: {mask.float().mean().item():.2f}')
W_obs = W_gt * mask.float()

# Overparameterised factorisation with r_model >> r_true
r_model = 20
A = torch.nn.Parameter(torch.randn(m, r_model) * 0.01)
B = torch.nn.Parameter(torch.randn(r_model, n) * 0.01)

opt = torch.optim.Adam([A, B], lr=1e-2)
losses, eranks = [], []

for step in range(2000):
    W_pred = A @ B
    loss   = ((W_pred - W_gt)[mask]).pow(2).mean()
    opt.zero_grad(); loss.backward(); opt.step()

    if step % 100 == 0:
        with torch.no_grad():
            sv = torch.linalg.svdvals(W_pred)
            p  = sv / sv.sum()
            ent = -(p * torch.log(p + 1e-12)).sum()
            er  = torch.exp(ent).item()
        losses.append(loss.item())
        eranks.append(er)

print(f'Final loss     : {losses[-1]:.4e}')
print(f'Final eff rank : {eranks[-1]:.2f}  (true rank = {r_true}, model rank = {r_model})')

In [ ]:
# ── 6b. Plot: effective rank collapses toward true rank during training ────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

steps_x = list(range(0, 2000, 100))
axes[0].semilogy(steps_x, losses, color='#2a5757', lw=2)
axes[0].set_xlabel('step'); axes[0].set_ylabel('MSE loss (observed entries)')
axes[0].set_title('Training loss')

axes[1].plot(steps_x, eranks, color='#2a5757', lw=2)
axes[1].axhline(r_true, ls='--', color='#63aac9', label=f'true rank = {r_true}')
axes[1].set_xlabel('step'); axes[1].set_ylabel('effective rank')
axes[1].set_title('Effective rank of W = AB during training')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── 6c. Singular value spectrum of recovered matrix ──────────────────────────
with torch.no_grad():
    W_final = (A @ B).numpy()

_, sv_final, _ = np.linalg.svd(W_final, full_matrices=False)
_, sv_gt,    _ = np.linalg.svd(W_gt.numpy(), full_matrices=False)

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(range(1, 21), sv_final[:20], color='#2a5757', label='recovered W', alpha=0.85)
ax.bar(range(1, 21), sv_gt[:20],    color='#63aac9', label='ground truth W', alpha=0.6, width=0.4)
ax.set_xlabel('singular value index'); ax.set_ylabel('sigma_i')
ax.set_title('Singular values: recovered vs ground truth')
ax.legend()
plt.tight_layout()
plt.show()

# Frobenius recovery error
print(f'Recovery Frobenius error: {np.linalg.norm(W_final - W_gt.numpy(), "fro"):.4f}')
print(f'Ground truth Frobenius norm: {np.linalg.norm(W_gt.numpy(), "fro"):.4f}')

**Reflection 6a**: The effective rank of $W = AB$ collapses toward the true rank even though no explicit rank constraint was imposed. Gradient descent with small-scale initialisation and the factored parameterisation *implicitly* regularises toward low nuclear norm (Gunasekar et al. 2017). How does this explain why LoRA fine-tuning ($\Delta W = AB$) works so well with small $r$, even without an explicit low-rank loss?

---
## 7 · Capstone — Truncated SVD as a Compression Codec

Apply everything to compress the full 400 000 × 50 GloVe matrix and measure the trade-off between compression ratio and semantic quality.

In [ ]:
# ── 7a. Truncated SVD of the full GloVe matrix ───────────────────────────────
# We use the 50d matrix (already 50 columns), so rank k <= 50
E_c = E - E.mean(axis=0, keepdims=True)   # mean-centre full matrix

# Economy SVD: dominant cost is O(n * d^2) = O(400000 * 50^2) — fast
U_full, S_full, Vh_full = np.linalg.svd(E_c, full_matrices=False)
print(f'Full SVD shapes: U={U_full.shape}, S={S_full.shape}, Vh={Vh_full.shape}')

# Compression ratio: original = 400000*50 floats; rank-k = 400000*k + k + k*50 floats
for k in [5, 10, 20, 50]:
    orig   = E.shape[0] * E.shape[1]
    compressed = E.shape[0] * k + k + k * E.shape[1]  # U_k, S_k, Vh_k
    ratio  = orig / compressed
    print(f'k={k:2d}: compression ratio = {ratio:.2f}x  ({compressed/1e6:.1f}M vs {orig/1e6:.1f}M floats)')

In [ ]:
# ── 7b. Semantic quality at each compression level ───────────────────────────
E_mean = E.mean(axis=0)

def analogy_accuracy(U, S, Vh, mean, sample_size=200):
    """Test king-man+woman=queen analogy on a compressed embedding."""
    E_k = U * S @ Vh + mean   # reconstruct
    E_k_unit = E_k / np.linalg.norm(E_k, axis=1, keepdims=True)

    correct = 0
    tests = [
        ('king', 'man', 'woman', 'queen'),
        ('paris', 'france', 'germany', 'berlin'),
        ('uncle', 'man', 'woman', 'aunt'),
        ('tall', 'taller', 'smaller', 'small'),
    ]
    for a, b, c, d in tests:
        if not all(w in word2idx for w in [a, b, c, d]):
            continue
        query = E_k[word2idx[a]] - E_k[word2idx[b]] + E_k[word2idx[c]]
        q_unit = query / np.linalg.norm(query)
        scores = E_k_unit @ q_unit
        scores[[word2idx[a], word2idx[b], word2idx[c]]] = -np.inf
        pred = words[np.argmax(scores)]
        correct += int(pred == d)
    return correct, len(tests)

print(f'{"k":5s}  analogy accuracy')
for k in [1, 5, 10, 25, 50]:
    c, t = analogy_accuracy(U_full[:, :k], S_full[:k], Vh_full[:k], E_mean)
    print(f'k={k:2d}    {c}/{t}')

**Reflection 7a**: At what rank do analogies start succeeding? The fact that they work at all for small $k$ reveals that semantic structure is **concentrated in the top singular directions**. How does this relate to the assumption underlying word2vec and BERT: that meaning can be represented in a continuous, low-dimensional space?

---
## Summary

| Concept | What you did |
|---|---|
| **Full SVD** | Factored a GloVe slice; read off singular values as the "energy" in each direction |
| **Low-rank approximation** | Eckart–Young theorem in practice; saw how rank controls reconstruction error |
| **PCA from scratch** | Showed PCA = SVD of mean-centred data; matched sklearn to machine precision |
| **Effective rank** | Measured soft rank via singular-value entropy; trained matrices have lower erank |
| **Framework APIs** | `torch.linalg.svd` (returns Vh) vs `tf.linalg.svd` (returns V) — API shapes differ |
| **Implicit regularisation** | Gradient descent on $W=AB$ collapses effective rank toward the true data rank |
| **Capstone** | Compressed GloVe; analogy quality degrades gracefully — meaning is low-dimensional |

**Connection to the course**: SVD is the backbone of PCA, LSA, spectral methods, recommender systems, and LoRA fine-tuning. Every time a paper says "the weight matrix lies in a low-dimensional subspace", they are pointing at the same spectrum you visualised here.